# XTSF: synthetic forecasting explanations with XPC

This is the project's single end-to-end notebook. It runs from Google
Drive/Colab or locally, builds a cached synthetic forecasting dataset, runs
grouped Monte Carlo Shapley explanations, and plots heightened positive
contribution parts in the style of the original archived XPC work.

## 1. Runtime and project paths

In [ ]:
from pathlib import Path
import sys

on_drive = False

if on_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    project_path = Path("/content/drive/MyDrive/Recherche/Thèse Gaspard/Codes/xtsf")
    data_path = Path("/content/drive/MyDrive/Recherche/Thèse Gaspard/Datasets")
else:
    project_path = Path.cwd().resolve()
    if project_path.name == "src":
        project_path = project_path.parent
    data_path = project_path / "datasets"

if not (project_path / "src" / "xpc").exists():
    raise FileNotFoundError(
        "Could not find src/xpc. Set project_path to the xtsf repository."
    )

output_path = project_path / "outputs"
output_path.mkdir(parents=True, exist_ok=True)
project_src = str(project_path / "src")
%cd $project_src
sys.path.insert(0, project_src)

print("Project root:", project_path)
print("Dataset path:", data_path)
print("Output path:", output_path)

## 2. Imports and configuration

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from xpc import BaselineMasker, FeatureGroups, ShapleyExplainer, TimeSeriesTensorSpec

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

SEED = 7
REBUILD_SYNTHETIC = False
DATASET_PATH = output_path / "synthetic_forecasting_panel.csv"

FEATURE_COLUMNS = [
    "heating_degree",
    "cooling_degree",
    "daily_peak",
    "weekday_work",
    "trend",
    "irrelevant",
]
GROUP_NAMES = ["weather", "calendar", "trend", "irrelevant"]
BASE_LOAD = 60.0
COEFFICIENTS = {
    "heating_degree": 1.20,
    "cooling_degree": 1.80,
    "daily_peak": 14.0,
    "weekday_work": 6.0,
    "trend": 10.0,
}

## 3. Synthetic dataset generation

In [ ]:
def build_synthetic_forecasting_dataset(n_days=120, seed=7):
    rng = np.random.default_rng(seed)
    n = 24 * int(n_days)
    timestamps = pd.date_range("2024-01-01", periods=n, freq="h")
    t = np.arange(n, dtype=float)

    hour = timestamps.hour.to_numpy()
    dayofweek = timestamps.dayofweek.to_numpy()
    annual_phase = 2.0 * np.pi * t / (24.0 * 365.0)
    daily_phase = 2.0 * np.pi * hour / 24.0

    temperature = (
        13.0
        + 9.0 * np.sin(annual_phase - 1.0)
        + 3.0 * np.sin(daily_phase - 0.4)
        + rng.normal(0.0, 1.1, size=n)
    )
    heating_degree = np.maximum(18.0 - temperature, 0.0)
    cooling_degree = np.maximum(temperature - 23.0, 0.0)

    # Positive basis functions: zero means absent under the baseline masker.
    evening_peak = np.exp(-0.5 * ((hour - 19.0) / 3.2) ** 2)
    morning_peak = 0.45 * np.exp(-0.5 * ((hour - 7.0) / 2.4) ** 2)
    daily_peak = evening_peak + morning_peak
    weekday_work = (dayofweek < 5).astype(float)
    trend = np.linspace(0.0, 1.0, n)
    irrelevant = rng.normal(0.0, 1.0, size=n)

    effect_weather = (
        COEFFICIENTS["heating_degree"] * heating_degree
        + COEFFICIENTS["cooling_degree"] * cooling_degree
    )
    effect_calendar = (
        COEFFICIENTS["daily_peak"] * daily_peak
        + COEFFICIENTS["weekday_work"] * weekday_work
    )
    effect_trend = COEFFICIENTS["trend"] * trend
    effect_irrelevant = np.zeros(n)
    load = BASE_LOAD + effect_weather + effect_calendar + effect_trend

    return pd.DataFrame(
        {
            "timestamp": timestamps,
            "temperature": temperature,
            "heating_degree": heating_degree,
            "cooling_degree": cooling_degree,
            "daily_peak": daily_peak,
            "weekday_work": weekday_work,
            "trend": trend,
            "irrelevant": irrelevant,
            "load": load,
            "effect_weather": effect_weather,
            "effect_calendar": effect_calendar,
            "effect_trend": effect_trend,
            "effect_irrelevant": effect_irrelevant,
        }
    )


if REBUILD_SYNTHETIC or not DATASET_PATH.exists():
    df = build_synthetic_forecasting_dataset(seed=SEED)
    df.to_csv(DATASET_PATH, index=False)
    print("Built synthetic dataset:", DATASET_PATH)
else:
    df = pd.read_csv(DATASET_PATH, parse_dates=["timestamp"])
    print("Loaded synthetic dataset:", DATASET_PATH)

df.head()

## 4. Dataset visualization

In [ ]:
def plot_series_overview(frame, days=21):
    view = frame.iloc[: 24 * days]
    fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
    axes[0].plot(view["timestamp"], view["load"], color="black", label="load")
    axes[0].set_ylabel("Load")
    axes[0].legend(loc="upper left")

    axes[1].plot(view["timestamp"], view["temperature"], label="temperature")
    axes[1].plot(view["timestamp"], view["heating_degree"], label="heating_degree")
    axes[1].plot(view["timestamp"], view["cooling_degree"], label="cooling_degree")
    axes[1].set_ylabel("Weather")
    axes[1].legend(loc="upper left", ncol=3)

    axes[2].plot(view["timestamp"], view["daily_peak"], label="daily_peak")
    axes[2].plot(view["timestamp"], view["weekday_work"], label="weekday_work")
    axes[2].plot(view["timestamp"], view["trend"], label="trend")
    axes[2].set_ylabel("Covariates")
    axes[2].legend(loc="upper left", ncol=3)
    axes[2].set_xlabel("Timestamp")
    fig.suptitle("Synthetic forecasting dataset")
    fig.tight_layout()
    plt.show()


def plot_seasonality(frame):
    hourly = frame.groupby(frame["timestamp"].dt.hour)["load"].mean()
    weekly = frame.groupby(frame["timestamp"].dt.dayofweek)["load"].mean()
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    axes[0].plot(hourly.index, hourly.values, marker="o")
    axes[0].set_title("Mean load by hour")
    axes[0].set_xlabel("Hour")
    axes[0].set_ylabel("Load")
    axes[1].bar(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], weekly.values)
    axes[1].set_title("Mean load by day of week")
    axes[1].set_ylabel("Load")
    fig.tight_layout()
    plt.show()


plot_series_overview(df)
plot_seasonality(df)

## 5. Forecasting model and grouped Shapley explanation

In [ ]:
def forecasting_model(x):
    x = np.asarray(x, dtype=float)
    heating = x[..., 0]
    cooling = x[..., 1]
    daily = x[..., 2]
    weekday = x[..., 3]
    trend = x[..., 4]
    forecast = (
        BASE_LOAD
        + COEFFICIENTS["heating_degree"] * heating
        + COEFFICIENTS["cooling_degree"] * cooling
        + COEFFICIENTS["daily_peak"] * daily
        + COEFFICIENTS["weekday_work"] * weekday
        + COEFFICIENTS["trend"] * trend
    )
    return forecast


feature_groups = FeatureGroups(
    {
        "weather": ["heating_degree", "cooling_degree"],
        "calendar": ["daily_peak", "weekday_work"],
        "trend": ["trend"],
        "irrelevant": ["irrelevant"],
    },
    remaining="ignore",
)

START = 24 * 28
LENGTH = 24 * 14
window = df.iloc[START : START + LENGTH].copy()
X = window[FEATURE_COLUMNS].to_numpy(dtype=float)[None, :, :]

explainer = ShapleyExplainer(
    forecasting_model,
    BaselineMasker(0.0),
    data_spec=TimeSeriesTensorSpec(feature_names=FEATURE_COLUMNS),
    feature_groups=feature_groups,
    n_coalitions=16,
    random_state=SEED,
    heighten=True,
)
explanation = explainer(X)

raw = explanation.values[0, :, 0, :]
parts = explanation.heightened.parts[0, :, 0, :]
pred = explanation.predictions[0, :, 0]
group_names = list(explanation.group_names)

true_effects = window[
    ["effect_weather", "effect_calendar", "effect_trend", "effect_irrelevant"]
].to_numpy(dtype=float)

print("values shape:", explanation.values.shape)
print("groups:", group_names)
print("max efficiency residual:", float(np.max(np.abs(explanation.efficiency_residual))))
print("max raw contribution error:", float(np.max(np.abs(raw - true_effects))))
print("max heightened sum error:", float(np.max(np.abs(parts.sum(axis=-1) - pred))))

## 6. Heightened contribution plots

In [ ]:
def plot_heightened_contributions(times, parts, pred, actual, group_names):
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.stackplot(times, parts.T, labels=group_names, alpha=0.85)
    ax.plot(times, pred, color="black", linewidth=1.8, label="model forecast")
    ax.plot(times, actual, color="white", linewidth=0.8, linestyle="--", label="synthetic target")
    ax.set_title("Heightened positive contribution parts")
    ax.set_ylabel("Load allocation")
    ax.set_xlabel("Timestamp")
    ax.legend(loc="upper left", ncol=3)
    fig.tight_layout()
    plt.show()


def plot_raw_contributions(times, raw, group_names):
    fig, ax = plt.subplots(figsize=(14, 4))
    for i, name in enumerate(group_names):
        ax.plot(times, raw[:, i], label=name)
    ax.set_title("Signed raw Shapley values before heightening")
    ax.set_ylabel("Contribution to forecast above baseline")
    ax.set_xlabel("Timestamp")
    ax.legend(loc="upper left", ncol=4)
    fig.tight_layout()
    plt.show()


def plot_single_timestamp(times, parts, raw, pred, group_names, index=24):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(group_names, raw[index])
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].set_title(f"Raw values at {times.iloc[index]}")
    axes[0].set_ylabel("Signed contribution")

    axes[1].bar(group_names, parts[index])
    axes[1].axhline(pred[index], color="black", linestyle="--", label="forecast")
    axes[1].set_title("Heightened parts")
    axes[1].set_ylabel("Positive allocation")
    axes[1].legend()
    fig.tight_layout()
    plt.show()


times = window["timestamp"].reset_index(drop=True)
actual = window["load"].to_numpy(dtype=float)
plot_heightened_contributions(times, parts, pred, actual, group_names)
plot_raw_contributions(times, raw, group_names)
plot_single_timestamp(times, parts, raw, pred, group_names, index=36)

## 7. Disable heightening when only signed values are needed

In [ ]:
signed_only = ShapleyExplainer(
    forecasting_model,
    BaselineMasker(0.0),
    data_spec=TimeSeriesTensorSpec(feature_names=FEATURE_COLUMNS),
    feature_groups=feature_groups,
    n_coalitions=16,
    random_state=SEED,
    heighten=False,
)(X)

print("heightened attached:", signed_only.heightened is not None)
print(
    "max signed reconstruction error:",
    float(np.max(np.abs(signed_only.base_values + signed_only.values.sum(axis=-1) - signed_only.predictions))),
)

## 8. Convergence against known contributions

The synthetic model is additive, so the columns `effect_weather`,
`effect_calendar`, `effect_trend`, and `effect_irrelevant` are an exact
structural decomposition. The benchmark compares those true per-sample
contributions with raw Shapley values plus the corresponding background
mean contribution, mirroring the expectation allocation used in the
original XPC pipeline.

Here, *interventional* (normal) Shapley uses unconditional empirical rows,
whereas *conditional* Shapley uses nearest empirical neighbors. Conditional
Shapley has a different estimand when groups are dependent, so we report MSE
both against the structural decomposition and against each method's exact
population target. The deterministic zero-baseline explanation above is an
oracle sanity check and is exact for every coalition budget.

In [ ]:
from itertools import combinations
from math import comb

from xpc import (
    ConditionalMasker, DataSpec, EmpiricalConditioner, RandomMasker,
)

EFFECT_COLUMNS = [
    "effect_weather",
    "effect_calendar",
    "effect_trend",
    "effect_irrelevant",
]
N_BENCHMARK_POINTS = 6
N_REPETITIONS = 5
CONDITIONAL_NEIGHBORS = 64
COALITION_SWEEP = [1, 2, 4, 8, 16, 32]
MASK_SAMPLE_SWEEP = [1, 2, 4, 8, 16, 32]
FIXED_COALITIONS = 16
FIXED_MASK_SAMPLES = 16

benchmark_background = df[FEATURE_COLUMNS].to_numpy(dtype=float)
benchmark_truth_all = df[EFFECT_COLUMNS].to_numpy(dtype=float)
benchmark_indexes = np.linspace(
    0, len(benchmark_background) - 1, N_BENCHMARK_POINTS, dtype=int
)
benchmark_X = benchmark_background[benchmark_indexes]
benchmark_truth = benchmark_truth_all[benchmark_indexes]
background_mean_parts = benchmark_truth_all.mean(axis=0)
benchmark_spec = DataSpec(feature_names=FEATURE_COLUMNS)
resolved_groups = feature_groups.resolve(len(FEATURE_COLUMNS), FEATURE_COLUMNS)


def features_for_players(players):
    features = []
    for player in players:
        features.extend(resolved_groups.groups[player])
    return tuple(dict.fromkeys(features))


def exact_empirical_conditional_value(x, players):
    present = features_for_players(players)
    if present:
        columns = np.asarray(present, dtype=int)
        scale = np.std(benchmark_background[:, columns], axis=0)
        scale = np.where(scale > 0, scale, 1.0)
        distances = np.sum(
            ((benchmark_background[:, columns] - x[columns]) / scale) ** 2,
            axis=1,
        )
        count = min(CONDITIONAL_NEIGHBORS, len(benchmark_background))
        indexes = np.argpartition(distances, count - 1)[:count]
        rows = benchmark_background[indexes].copy()
        rows[:, columns] = x[columns]
    else:
        rows = benchmark_background
    return float(np.mean(forecasting_model(rows)))


def exact_empirical_conditional_shapley(x):
    n_players = resolved_groups.n_players
    cache = {}

    def value(players):
        key = tuple(sorted(players))
        if key not in cache:
            cache[key] = exact_empirical_conditional_value(x, key)
        return cache[key]

    result = np.zeros(n_players, dtype=float)
    for player in range(n_players):
        others = [item for item in range(n_players) if item != player]
        for size in range(n_players):
            weight = 1.0 / (n_players * comb(n_players - 1, size))
            for coalition in combinations(others, size):
                result[player] += weight * (
                    value((*coalition, player)) - value(coalition)
                )
    return result


conditional_reference = np.stack([
    exact_empirical_conditional_shapley(x) for x in benchmark_X
]) + background_mean_parts
method_references = {
    "interventional": benchmark_truth,
    "conditional": conditional_reference,
}

print("benchmark points:", benchmark_X.shape[0])
print(
    "conditional estimand gap MSE:",
    float(np.mean((conditional_reference - benchmark_truth) ** 2)),
)

In [ ]:
def estimate_benchmark_parts(method, n_coalitions, n_mask_samples, seed):
    if method == "interventional":
        masker = RandomMasker(benchmark_background)
    elif method == "conditional":
        masker = ConditionalMasker(
            benchmark_background,
            EmpiricalConditioner(n_neighbors=CONDITIONAL_NEIGHBORS),
        )
    else:
        raise ValueError(f"Unknown method: {method}")

    explanation = ShapleyExplainer(
        forecasting_model,
        masker,
        data_spec=benchmark_spec,
        feature_groups=feature_groups,
        n_coalitions=n_coalitions,
        n_mask_samples=n_mask_samples,
        random_state=seed,
        heighten=False,
    )(benchmark_X)
    return explanation.values[:, 0, :] + background_mean_parts


def run_benchmark_setting(axis, budget, n_coalitions, n_mask_samples):
    rows = []
    for method_index, method in enumerate(method_references):
        for repetition in range(N_REPETITIONS):
            seed = (
                SEED
                + 100_000 * method_index
                + 1_000 * budget
                + repetition
            )
            estimated = estimate_benchmark_parts(
                method, n_coalitions, n_mask_samples, seed
            )
            rows.append(
                {
                    "axis": axis,
                    "budget": budget,
                    "method": method,
                    "repetition": repetition,
                    "mse_structural": float(
                        np.mean((estimated - benchmark_truth) ** 2)
                    ),
                    "mse_estimand": float(
                        np.mean((estimated - method_references[method]) ** 2)
                    ),
                }
            )
    return rows


benchmark_rows = []
for n_coalitions in COALITION_SWEEP:
    benchmark_rows.extend(
        run_benchmark_setting(
            "coalitions",
            n_coalitions,
            n_coalitions,
            FIXED_MASK_SAMPLES,
        )
    )
for n_mask_samples in MASK_SAMPLE_SWEEP:
    benchmark_rows.extend(
        run_benchmark_setting(
            "mask_samples",
            n_mask_samples,
            FIXED_COALITIONS,
            n_mask_samples,
        )
    )

benchmark_results = pd.DataFrame(benchmark_rows)
benchmark_summary = (
    benchmark_results.groupby(["axis", "budget", "method"], as_index=False)
    .agg(
        mse_structural_mean=("mse_structural", "mean"),
        mse_structural_std=("mse_structural", "std"),
        mse_estimand_mean=("mse_estimand", "mean"),
        mse_estimand_std=("mse_estimand", "std"),
    )
)
benchmark_results.to_csv(output_path / "shapley_convergence_runs.csv", index=False)
benchmark_summary.to_csv(output_path / "shapley_convergence_summary.csv", index=False)
benchmark_summary

In [ ]:
from matplotlib.ticker import ScalarFormatter


def plot_benchmark_panel(ax, axis_name, mean_column, std_column, title):
    subset = benchmark_summary[benchmark_summary["axis"] == axis_name]
    for method in ["interventional", "conditional"]:
        values = subset[subset["method"] == method].sort_values("budget")
        ax.errorbar(
            values["budget"],
            np.maximum(values[mean_column], np.finfo(float).tiny),
            yerr=values[std_column],
            marker="o",
            capsize=3,
            label=method,
        )
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xticks(subset["budget"].unique())
    ax.get_xaxis().set_major_formatter(ScalarFormatter())
    ax.set_title(title)
    ax.set_xlabel(
        "Number of coalitions" if axis_name == "coalitions"
        else "Samples per coalition"
    )
    ax.set_ylabel("Contribution MSE")
    ax.grid(alpha=0.25, which="both")
    ax.legend()


fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
plot_benchmark_panel(
    axes[0, 0], "coalitions", "mse_structural_mean",
    "mse_structural_std", "Against structural truth",
)
plot_benchmark_panel(
    axes[0, 1], "mask_samples", "mse_structural_mean",
    "mse_structural_std", "Against structural truth",
)
plot_benchmark_panel(
    axes[1, 0], "coalitions", "mse_estimand_mean",
    "mse_estimand_std", "Against each method's estimand",
)
plot_benchmark_panel(
    axes[1, 1], "mask_samples", "mse_estimand_mean",
    "mse_estimand_std", "Against each method's estimand",
)
fig.suptitle("Monte Carlo Shapley convergence")
fig.savefig(output_path / "shapley_convergence.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Aggregation modes from the original XPC experiments

The archived XPC report compared three definitions for reducing raw-feature
Shapley values to `climate` and `non_climate` contributions:

1. **Default / post-hoc:** compute the six raw-feature Shapley values, then sum them.
2. **Coalitional Shapley (`agg 1`):** for each macro-group $C$, merge $C$ into one player while every feature outside $C$ remains an individual player.
3. **Simplified Shapley (`agg 2`):** make `climate` and `non_climate` the only two players, so each contribution needs only the two marginal terms of a two-player game.

These are different games, not merely faster implementations. The experiment
therefore compares estimates both with the structural synthetic contributions
and with the exact all-coalition estimand for the same aggregation mode and
masking rule. The fixed Monte Carlo budget mirrors the archived aggregation
experiment. `agg 2` has no coalition-sampling parameter because its two-player
formula is already exhaustive.


In [ ]:
AGGREGATION_GROUPS = {
    "climate": (0, 1),
    "non_climate": (2, 3, 4, 5),
}
AGGREGATION_MODES = ("default", "coalitional", "simplified")
AGGREGATION_METHODS = ("interventional", "conditional")
aggregation_truth = np.column_stack([
    benchmark_truth[:, 0],
    benchmark_truth[:, 1:].sum(axis=1),
])
aggregation_background_parts = np.array([
    background_mean_parts[0],
    background_mean_parts[1:].sum(),
])


def exact_aggregation_value(method, x, present):
    present = tuple(sorted(present))
    if method == "interventional" or not present:
        rows = benchmark_background.copy()
    elif method == "conditional":
        columns = np.asarray(present, dtype=int)
        scale = np.std(benchmark_background[:, columns], axis=0)
        scale = np.where(scale > 0, scale, 1.0)
        distances = np.sum(
            ((benchmark_background[:, columns] - x[columns]) / scale) ** 2,
            axis=1,
        )
        count = min(CONDITIONAL_NEIGHBORS, len(benchmark_background))
        neighbors = np.argpartition(distances, count - 1)[:count]
        rows = benchmark_background[neighbors].copy()
    else:
        raise ValueError(f"Unknown method: {method}")
    if present:
        columns = np.asarray(present, dtype=int)
        rows[:, columns] = x[columns]
    return float(np.mean(forecasting_model(rows)))


def exact_aggregation_contributions(mode, method, x):
    cache = {}

    def value(present):
        key = tuple(sorted(present))
        if key not in cache:
            cache[key] = exact_aggregation_value(method, x, key)
        return cache[key]

    p = len(FEATURE_COLUMNS)
    if mode == "default":
        feature_values = np.zeros(p)
        for feature in range(p):
            others = [item for item in range(p) if item != feature]
            for size in range(p):
                weight = 1.0 / (p * comb(p - 1, size))
                for coalition in combinations(others, size):
                    feature_values[feature] += weight * (
                        value((*coalition, feature)) - value(coalition)
                    )
        return np.array([
            feature_values[list(AGGREGATION_GROUPS[name])].sum()
            for name in AGGREGATION_GROUPS
        ])

    if mode == "coalitional":
        result = []
        for group in AGGREGATION_GROUPS.values():
            others = [item for item in range(p) if item not in group]
            n_players = len(others) + 1
            contribution = 0.0
            for size in range(len(others) + 1):
                weight = 1.0 / (n_players * comb(len(others), size))
                for coalition in combinations(others, size):
                    contribution += weight * (
                        value((*coalition, *group)) - value(coalition)
                    )
            result.append(contribution)
        return np.asarray(result)

    if mode == "simplified":
        climate = AGGREGATION_GROUPS["climate"]
        non_climate = AGGREGATION_GROUPS["non_climate"]
        empty, full = value(()), value(range(p))
        climate_only, non_climate_only = value(climate), value(non_climate)
        return 0.5 * np.array([
            (climate_only - empty) + (full - non_climate_only),
            (non_climate_only - empty) + (full - climate_only),
        ])
    raise ValueError(f"Unknown aggregation mode: {mode}")


aggregation_references = {
    (method, mode): np.stack([
        exact_aggregation_contributions(mode, method, x)
        for x in benchmark_X
    ]) + aggregation_background_parts
    for method in AGGREGATION_METHODS
    for mode in AGGREGATION_MODES
}
aggregation_exact_summary = pd.DataFrame([
    {
        "method": method,
        "mode": mode,
        "mse_structural_exact": float(np.mean(
            (aggregation_references[(method, mode)] - aggregation_truth) ** 2
        )),
        "efficiency_rmse_exact": float(np.sqrt(np.mean(
            (aggregation_references[(method, mode)].sum(axis=1)
             - aggregation_truth.sum(axis=1)) ** 2
        ))),
    }
    for method in AGGREGATION_METHODS
    for mode in AGGREGATION_MODES
])
aggregation_exact_summary.to_csv(
    output_path / "aggregation_exact_estimands.csv", index=False
)
aggregation_exact_summary


In [ ]:
from time import perf_counter

N_AGGREGATION_COALITIONS = 16
N_AGGREGATION_MASK_SAMPLES = 16
N_AGGREGATION_REPETITIONS = 5


def make_aggregation_masker(method):
    if method == "interventional":
        return RandomMasker(benchmark_background)
    if method == "conditional":
        return ConditionalMasker(
            benchmark_background,
            EmpiricalConditioner(n_neighbors=CONDITIONAL_NEIGHBORS),
        )
    raise ValueError(f"Unknown method: {method}")


def sampled_aggregation_value(masker, x, present, n_samples, rng, unit):
    rows = masker.mask(x, tuple(present), n_samples, rng, unit=unit)
    return float(np.mean(forecasting_model(rows)))


def sampled_group_marginal(
    masker, x, group, other_players, n_coalitions, n_samples, rng, unit
):
    deltas = []
    for _ in range(n_coalitions):
        size = int(rng.integers(0, len(other_players) + 1))
        selected = (
            rng.choice(len(other_players), size=size, replace=False)
            if size else []
        )
        present = tuple(
            feature
            for player in selected
            for feature in other_players[int(player)]
        )
        before = sampled_aggregation_value(
            masker, x, present, n_samples, rng, unit
        )
        after = sampled_aggregation_value(
            masker, x, (*present, *group), n_samples, rng, unit
        )
        deltas.append(after - before)
    return float(np.mean(deltas))


def estimate_aggregation_mode(method, mode, n_coalitions, n_samples, seed):
    rng = np.random.default_rng(seed)
    masker = make_aggregation_masker(method)
    masker.prepare(benchmark_X, benchmark_spec)
    estimates = []
    started = perf_counter()
    for row_index, x in enumerate(benchmark_X):
        unit = (row_index,)
        if mode == "default":
            feature_values = []
            for feature in range(len(FEATURE_COLUMNS)):
                others = [
                    (item,) for item in range(len(FEATURE_COLUMNS))
                    if item != feature
                ]
                feature_values.append(sampled_group_marginal(
                    masker, x, (feature,), others, n_coalitions,
                    n_samples, rng, unit,
                ))
            estimates.append([
                np.sum(np.asarray(feature_values)[list(group)])
                for group in AGGREGATION_GROUPS.values()
            ])
            value_calls = 2 * len(FEATURE_COLUMNS) * n_coalitions
        elif mode == "coalitional":
            row = []
            for group in AGGREGATION_GROUPS.values():
                others = [
                    (item,) for item in range(len(FEATURE_COLUMNS))
                    if item not in group
                ]
                row.append(sampled_group_marginal(
                    masker, x, group, others, n_coalitions,
                    n_samples, rng, unit,
                ))
            estimates.append(row)
            value_calls = 2 * len(AGGREGATION_GROUPS) * n_coalitions
        elif mode == "simplified":
            climate = AGGREGATION_GROUPS["climate"]
            non_climate = AGGREGATION_GROUPS["non_climate"]
            empty = sampled_aggregation_value(masker, x, (), n_samples, rng, unit)
            climate_only = sampled_aggregation_value(
                masker, x, climate, n_samples, rng, unit
            )
            non_climate_only = sampled_aggregation_value(
                masker, x, non_climate, n_samples, rng, unit
            )
            full = sampled_aggregation_value(
                masker, x, range(len(FEATURE_COLUMNS)), n_samples, rng, unit
            )
            estimates.append(0.5 * np.array([
                (climate_only - empty) + (full - non_climate_only),
                (non_climate_only - empty) + (full - climate_only),
            ]))
            value_calls = 4
        else:
            raise ValueError(f"Unknown aggregation mode: {mode}")
    elapsed = perf_counter() - started
    return (
        np.asarray(estimates) + aggregation_background_parts,
        elapsed / len(benchmark_X),
        value_calls * n_samples,
    )


aggregation_rows = []
for method_index, method in enumerate(AGGREGATION_METHODS):
    for mode_index, mode in enumerate(AGGREGATION_MODES):
        reference = aggregation_references[(method, mode)]
        for repetition in range(N_AGGREGATION_REPETITIONS):
            seed = SEED + 100_000 * method_index + 10_000 * mode_index + repetition
            estimated, seconds_per_point, model_rows_per_point = (
                estimate_aggregation_mode(
                    method, mode, N_AGGREGATION_COALITIONS,
                    N_AGGREGATION_MASK_SAMPLES, seed,
                )
            )
            aggregation_rows.append({
                "method": method,
                "mode": mode,
                "repetition": repetition,
                "mse_structural": float(np.mean(
                    (estimated - aggregation_truth) ** 2
                )),
                "mse_estimand": float(np.mean((estimated - reference) ** 2)),
                "efficiency_rmse": float(np.sqrt(np.mean(
                    (estimated.sum(axis=1) - aggregation_truth.sum(axis=1)) ** 2
                ))),
                "seconds_per_point": seconds_per_point,
                "model_rows_per_point": model_rows_per_point,
            })

aggregation_results = pd.DataFrame(aggregation_rows)
aggregation_summary = (
    aggregation_results.groupby(["method", "mode"], as_index=False)
    .agg(
        mse_structural_mean=("mse_structural", "mean"),
        mse_structural_std=("mse_structural", "std"),
        mse_estimand_mean=("mse_estimand", "mean"),
        mse_estimand_std=("mse_estimand", "std"),
        efficiency_rmse_mean=("efficiency_rmse", "mean"),
        seconds_per_point_mean=("seconds_per_point", "mean"),
        seconds_per_point_std=("seconds_per_point", "std"),
        model_rows_per_point=("model_rows_per_point", "first"),
    )
)
aggregation_results.to_csv(output_path / "aggregation_mode_runs.csv", index=False)
aggregation_summary.to_csv(output_path / "aggregation_mode_summary.csv", index=False)
aggregation_summary


In [ ]:
def plot_aggregation_metric(
    ax, frame, value_column, title, ylabel, error_column=None, log=True
):
    x = np.arange(len(AGGREGATION_MODES), dtype=float)
    width = 0.36
    for method_index, method in enumerate(AGGREGATION_METHODS):
        values = (
            frame[frame["method"] == method]
            .set_index("mode").loc[list(AGGREGATION_MODES)]
        )
        heights = np.maximum(
            values[value_column].to_numpy(), np.finfo(float).tiny
        )
        errors = None
        if error_column is not None:
            errors = np.minimum(
                values[error_column].fillna(0).to_numpy(), 0.95 * heights
            )
        offset = (method_index - 0.5) * width
        ax.bar(x + offset, heights, width, yerr=errors, capsize=3, label=method)
    ax.set_xticks(x, AGGREGATION_MODES)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    if log:
        ax.set_yscale("log")
    ax.grid(axis="y", alpha=0.25, which="both")
    ax.legend()


fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
plot_aggregation_metric(
    axes[0, 0], aggregation_summary, "mse_structural_mean",
    "Monte Carlo error vs structural truth", "MSE",
    "mse_structural_std",
)
plot_aggregation_metric(
    axes[0, 1], aggregation_summary, "mse_estimand_mean",
    "Monte Carlo error vs exact mode estimand", "MSE",
    "mse_estimand_std",
)
plot_aggregation_metric(
    axes[1, 0], aggregation_exact_summary, "mse_structural_exact",
    "Exact estimand gap from structural truth", "MSE",
)
plot_aggregation_metric(
    axes[1, 1], aggregation_summary, "seconds_per_point_mean",
    "Measured computation time", "Seconds per explained point",
    "seconds_per_point_std",
)
fig.suptitle(
    f"Aggregation modes ({N_AGGREGATION_COALITIONS} coalitions, "
    f"{N_AGGREGATION_MASK_SAMPLES} samples/value)"
)
fig.savefig(output_path / "aggregation_modes.png", dpi=160, bbox_inches="tight")
plt.show()

aggregation_summary[[
    "method", "mode", "model_rows_per_point", "seconds_per_point_mean"
]]


### Reading the comparison

Because the synthetic forecasting model is additive, the three exact
interventional estimands coincide with structural truth; their remaining
errors come from Monte Carlo estimation. Conditional masking changes the
coalition-value function, so its three aggregation games need not agree. In
this fixed synthetic sample the exact structural MSE is about 11.40 for
post-hoc aggregation, 10.94 for Coalitional Shapley, and 10.20 for the
two-player Simplified value. This is an estimand comparison, not evidence
that the smallest number is generally the most causal definition.

At the configured budget, the masked model-row counts per explained point
are 3072, 1024, and 64 respectively. The large speedup reproduces the main
finding of the archived experiment. One correction is important: computing
each Coalitional group in its own reduced game does not generally guarantee
joint efficiency across the returned groups; the exact conditional run makes
that visible. Default aggregation and the single two-player game do retain
efficiency.
